# OpportunityMatch AI — Week 02
## Mini Semantic Search Engine for Student Opportunities

**Goal:** help students find relevant hackathons, internships, workshops, scholarships, fellowships, bootcamps, and projects using the meaning of a query—not only exact keyword matches.

This notebook documents the workflow:
1. Check the Python environment and required packages.
2. Load and inspect the dataset.
3. Check data quality and category distribution.
4. Preview text preparation and chunking.
5. Build/connect to the search index implemented in `task.py`.
6. Try semantic searches and category filters.
7. Evaluate ranking with Recall@3 and MRR.
8. Record limitations and future improvements.

The current dataset is a small synthetic demonstration dataset, not a live directory of verified opportunities.

## 1. Check the notebook's Python environment

Run this first to see exactly which Python kernel is active and whether the project packages are available.

In [1]:
import sys
from importlib.util import find_spec

print("Notebook Python:", sys.executable)
for package in ["pandas", "sentence_transformers", "chromadb", "langchain_text_splitters"]:
    print(f"{package}: {'available' if find_spec(package) else 'MISSING'}")

Notebook Python: c:\Users\tesni\OneDrive\Desktop\aws-ai-ml-track\.venv\Scripts\python.exe
pandas: available
sentence_transformers: MISSING
chromadb: available
langchain_text_splitters: available


## 2. Locate and load the dataset

This expects `data/opportunities.csv` to be inside the same `week-02` folder as this notebook.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

BASE_DIR = Path.cwd()
DATA_PATH = BASE_DIR / "data" / "opportunities.csv"
print("Working directory:", BASE_DIR)
print("Dataset path:", DATA_PATH)
print("Dataset exists:", DATA_PATH.exists())

if not DATA_PATH.exists():
    raise FileNotFoundError("Open/run this notebook from week-02 so data/opportunities.csv can be found.")

df = pd.read_csv(DATA_PATH)
required_columns = {"id", "title", "category", "description", "skills", "eligibility"}
missing_columns = required_columns - set(df.columns)

print(f"Rows and columns: {df.shape}")
print(f"Unique categories: {df['category'].nunique() if 'category' in df.columns else 'N/A'}")
print("Missing required columns:", sorted(missing_columns))
display(df.head())

Working directory: c:\Users\tesni\OneDrive\Desktop\aws-ai-ml-track\week-02
Dataset path: c:\Users\tesni\OneDrive\Desktop\aws-ai-ml-track\week-02\data\opportunities.csv
Dataset exists: True
Rows and columns: (20, 6)
Unique categories: 10
Missing required columns: []


,id,title,category,description,skills,eligibility
0,1,AI Innovation Hackathon,Hackathon,Build an artificial intelligence solution for ...,"Python, machine learning, teamwork, problem so...",University students and recent graduates
1,2,Python Machine Learning Workshop,Workshop,Learn machine learning fundamentals through pr...,"Python, pandas, NumPy, scikit-learn",Beginners with basic Python knowledge
2,3,Cybersecurity Challenge,Competition,Solve security challenges involving network de...,"Networking, Linux, cybersecurity, problem solving",Students interested in cybersecurity
3,4,Full Stack Web Development Bootcamp,Bootcamp,"Build responsive web applications using HTML, ...","HTML, CSS, JavaScript, React, Node.js, SQL",Beginner and intermediate developers
4,5,Data Science Internship,Internship,"Work with a data team to clean datasets, explo...","Python, pandas, SQL, statistics, data visualiz...","Students studying software engineering, comput..."


## 3. Data quality checks

Check missing values, duplicate IDs, blank text fields, and how many records are in each category.

In [2]:
print("Missing values by column:")
display(df.isna().sum().rename("missing_count").to_frame())

if "id" in df.columns:
    print("Duplicate IDs:", int(df["id"].duplicated().sum()))

for column in ["title", "description", "category", "skills", "eligibility"]:
    if column in df.columns:
        blanks = df[column].fillna("").astype(str).str.strip().eq("").sum()
        print(f"Blank {column} values: {int(blanks)}")

print("\nCategory distribution:")
display(df["category"].value_counts().rename_axis("category").to_frame("count"))

Missing values by column:


,missing_count
id,0
title,0
category,0
description,0
skills,0
eligibility,0


Duplicate IDs: 0
Blank title values: 0
Blank description values: 0
Blank category values: 0
Blank skills values: 0
Blank eligibility values: 0

Category distribution:


,count
category,
Workshop,4
Internship,4
Hackathon,2
Competition,2
Bootcamp,2
Scholarship,2
Fellowship,1
Program,1
Project,1


## 4. Prepare a combined text representation

The title, category, description, skills, and eligibility can all help a semantic model understand an opportunity. This preview is for inspection; the actual project processing remains in `task.py`.

In [3]:
TEXT_FIELDS = ["title", "category", "description", "skills", "eligibility"]

def combine_opportunity_text(row):
    parts = []
    for field in TEXT_FIELDS:
        value = row.get(field, "")
        if pd.notna(value) and str(value).strip():
            parts.append(f"{field.title()}: {str(value).strip()}")
    return "\n".join(parts)

preview = df.copy()
preview["combined_text"] = preview.apply(combine_opportunity_text, axis=1)
display(preview[["title", "category", "combined_text"]].head(3))

,title,category,combined_text
0,AI Innovation Hackathon,Hackathon,Title: AI Innovation Hackathon\nCategory: Hack...
1,Python Machine Learning Workshop,Workshop,Title: Python Machine Learning Workshop\nCateg...
2,Cybersecurity Challenge,Competition,Title: Cybersecurity Challenge\nCategory: Comp...


## 5. Load project functions and preview chunking

If this import fails, read the **first missing module name** in the error. It usually means the notebook kernel and terminal environment differ, or a dependency is missing.

In [4]:
from task import load_data, TextProcessor, OpportunitySearchEngine, evaluate, display_results

project_df = load_data(DATA_PATH)
print("Validated records:", len(project_df))

processor = TextProcessor()
sample_row = project_df.iloc[0].to_dict()
sample_chunks = processor.create_chunks(sample_row)
print("Chunks from first record:", len(sample_chunks))
for i, chunk in enumerate(sample_chunks[:3], start=1):
    print(f"\n--- Chunk {i} ---\n{chunk[:700]}")

ModuleNotFoundError: No module named 'sentence_transformers'

## 6. Build or connect to the semantic search index

The first successful run may download the Sentence Transformers model and create a local ChromaDB database. If the collection already exists, this cell reuses it; otherwise it indexes the dataset.

In [ ]:
engine = OpportunitySearchEngine()

try:
    engine.collection = engine.client.get_collection("student_opportunities")
    print("Connected to existing search collection.")
except Exception:
    print("No existing collection found. Indexing dataset...")
    summary = engine.ingest_data()
    print("Ingestion summary:", summary)
    engine.collection = engine.client.get_collection("student_opportunities")

print("Search index is ready.")

## 7. Try semantic search queries

Try natural-language requests that a student might type. Compare the result titles, categories, and scores.

In [ ]:
test_queries = [
    "I want to build software with a team and compete in a hackathon",
    "learning opportunities for a beginner interested in artificial intelligence",
    "internship where I can use Python and data analysis",
    "scholarship or fellowship for university students",
]

for query in test_queries:
    print("\n" + "=" * 80)
    print("QUERY:", query)
    results = engine.search(query, top_k=3)
    display_results(results)

## 8. Filter by category

Use a category that exists in the dataset. This example selects the first available category automatically.

In [ ]:
categories = sorted(project_df["category"].dropna().astype(str).unique())
print("Available categories:", categories)

query = "opportunity to improve programming skills"
chosen_category = categories[0]
print(f"\nQuery: {query}\nCategory filter: {chosen_category}")
filtered_results = engine.search(query, top_k=3, category=chosen_category)
display_results(filtered_results)

## 9. Evaluate ranking quality

Recall@3 measures whether relevant opportunities appear among the top three results. MRR rewards relevant results that appear closer to the top.

**Important:** verify the relevance labels in `task.py` manually. These metrics are only an initial baseline, especially with a small dataset.

In [ ]:
evaluation_results = evaluate(engine)
print("Evaluation finished. Verify the labels before reporting these metrics.")

## 10. Limitations and future improvements

- **Small dataset:** 20 records are suitable for a demonstration, not a production benchmark.
- **Relevance labels:** manually verify expected results for each evaluation query.
- **Ranking:** combine semantic similarity with keyword overlap, eligibility, freshness, and deadlines.
- **Data freshness:** store source links and closing dates to avoid recommending expired opportunities.
- **Personalization:** add filters for study level, location, skills, and eligibility.
- **Evaluation:** add more labeled queries and compare against a keyword-search baseline.

## Conclusion

OpportunityMatch AI demonstrates an end-to-end retrieval workflow: data validation, text preparation, embeddings, vector storage, semantic search, hybrid ranking, and evaluation. The next step is to measure ranking quality carefully and improve it using verified data and repeatable tests.
